In [1]:
!pip install yfinance duckdb streamlit plotly pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 66.0 MB/s eta 0:00:00


In [2]:
import yfinance as yf
import pandas as pd
from pathlib import Path
from datetime import datetime

TICKERS = ["AAPL", "MSFT", "GOOGL", "TSLA","NKE"]

def fetch_stock_data(tickers=TICKERS, period="1y"):
    Path("data/raw").mkdir(parents=True, exist_ok=True)
    data = {}
    for ticker in tickers:
        df = yf.download(ticker, period=period, auto_adjust=True, progress=False)

        # ✅ Fix: flatten MultiIndex columns if present
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.reset_index(inplace=True)
        df.columns = [str(c).strip() for c in df.columns]  # clean any weird names
        df["Ticker"] = ticker
        df.to_csv(f"data/raw/{ticker}_{datetime.today().date()}.csv", index=False)
        data[ticker] = df
        print(f"✅ {ticker}: {len(df)} rows | columns: {list(df.columns)}")
    return data

raw_data = fetch_stock_data()

✅ AAPL: 250 rows | columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker']
✅ MSFT: 250 rows | columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker']
✅ GOOGL: 250 rows | columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker']
✅ TSLA: 250 rows | columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker']
✅ NKE: 250 rows | columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker']


In [3]:
def add_indicators(df):
    df = df.copy().sort_values("Date")
    df["MA_20"] = df["Close"].rolling(20).mean()
    df["MA_50"] = df["Close"].rolling(50).mean()
    df["Daily_Return"] = df["Close"].pct_change()
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    df["RSI_14"] = 100 - (100 / (1 + gain / loss))
    df["Volatility_20"] = df["Daily_Return"].rolling(20).std()
    return df.dropna()

def transform_all(raw_data):
    combined = pd.concat([add_indicators(df) for df in raw_data.values()], ignore_index=True)
    print(f"✅ Transformed: {len(combined)} total rows")
    return combined

transformed_df = transform_all(raw_data)
transformed_df.head()

✅ Transformed: 1005 total rows


,Date,Close,High,Low,Open,Volume,Ticker,MA_20,MA_50,Daily_Return,RSI_14,Volatility_20
0,2025-06-25,200.948502,203.052101,200.011351,200.838835,39525700,AAPL,199.747651,201.958298,0.006291,47.849823,0.010628
1,2025-06-26,200.390182,202.025206,198.854861,200.818871,50799100,AAPL,199.776563,201.933279,-0.002778,50.668546,0.010650
2,2025-06-27,200.469955,202.603462,199.393230,201.277496,73188600,AAPL,199.832891,201.917422,0.000398,44.194628,0.010633
3,2025-06-30,204.547546,206.760812,198.655473,201.397130,91912800,AAPL,200.048235,202.139833,0.020340,57.131883,0.011513
4,2025-07-01,207.189514,209.552319,205.514603,206.042994,78788900,AAPL,200.353307,202.361120,0.012916,59.360253,0.011797


In [4]:
# Close any open DuckDB connections before reconnecting
try:
    con.close()
    print("✅ Closed existing connection")
except:
    print("ℹ️ No open connection found")

ℹ️ No open connection found


In [5]:
import duckdb
from pathlib import Path

DB_PATH = "data/warehouse.duckdb"
Path("data").mkdir(exist_ok=True)

def load_to_duckdb(df, table="stock_prices"):
    # ✅ 'with' block auto-closes the connection even if an error occurs
    with duckdb.connect(DB_PATH) as con:
        con.execute(f"DROP TABLE IF EXISTS {table}")
        con.execute(f"CREATE TABLE {table} AS SELECT * FROM df")
        count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"✅ Loaded {count} rows into DuckDB → '{table}'")

load_to_duckdb(transformed_df)

✅ Loaded 1005 rows into DuckDB → 'stock_prices'


In [6]:
con = duckdb.connect(DB_PATH, read_only=True)
print(con.execute("PRAGMA table_info('stock_prices')").df())
con.close()

    cid           name          type  notnull dflt_value     pk
0     0           Date  TIMESTAMP_NS    False       None  False
1     1          Close        DOUBLE    False       None  False
2     2           High        DOUBLE    False       None  False
3     3            Low        DOUBLE    False       None  False
4     4           Open        DOUBLE    False       None  False
5     5         Volume        BIGINT    False       None  False
6     6         Ticker       VARCHAR    False       None  False
7     7          MA_20        DOUBLE    False       None  False
8     8          MA_50        DOUBLE    False       None  False
9     9   Daily_Return        DOUBLE    False       None  False
10   10         RSI_14        DOUBLE    False       None  False
11   11  Volatility_20        DOUBLE    False       None  False


In [7]:
app_code = '''
import streamlit as st
import duckdb
import pandas as pd
import plotly.graph_objects as go

st.set_page_config(page_title="Stock Dashboard", layout="wide")
st.title("📈 Stock Market Analytics Pipeline")

@st.cache_data(ttl=3600)
def load_data():
    con = duckdb.connect("data/warehouse.duckdb", read_only=True)
    df = con.execute("SELECT * FROM stock_prices ORDER BY Date").df()
    con.close()
    return df

df = load_data()
ticker = st.sidebar.selectbox("Select Ticker", df["Ticker"].unique())
filtered = df[df["Ticker"] == ticker]

col1, col2, col3, col4 = st.columns(4)
latest = filtered.iloc[-1]
col1.metric("Close Price", f"${latest[\'Close\']:.2f}")
col2.metric("RSI (14)", f"{latest[\'RSI_14\']:.1f}")
col3.metric("Volatility", f"{latest[\'Volatility_20\']:.4f}")
col4.metric("Daily Return", f"{latest[\'Daily_Return\']*100:.2f}%")

fig = go.Figure()
fig.add_trace(go.Candlestick(x=filtered["Date"], open=filtered["Open"],
    high=filtered["High"], low=filtered["Low"], close=filtered["Close"], name="OHLC"))
fig.add_trace(go.Scatter(x=filtered["Date"], y=filtered["MA_20"], name="MA 20", line=dict(color="orange")))
fig.add_trace(go.Scatter(x=filtered["Date"], y=filtered["MA_50"], name="MA 50", line=dict(color="blue")))
fig.update_layout(title=f"{ticker} Price + Moving Averages", xaxis_rangeslider_visible=False)
st.plotly_chart(fig, use_container_width=True)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=filtered["Date"], y=filtered["RSI_14"], name="RSI"))
fig2.add_hline(y=70, line_dash="dash", line_color="red", annotation_text="Overbought")
fig2.add_hline(y=30, line_dash="dash", line_color="green", annotation_text="Oversold")
st.subheader("RSI (14)")
st.plotly_chart(fig2, use_container_width=True)
'''

with open("dashboard_app.py", "w") as f:
    f.write(app_code)
print("✅ dashboard_app.py written")

✅ dashboard_app.py written


In [8]:
import subprocess, time, re

# Kill any previous processes
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(2)

# Start Streamlit
proc = subprocess.Popen(
    ["streamlit", "run", "dashboard_app.py",
     "--server.port=8501",
     "--server.headless=true",
     "--server.enableCORS=false",
     "--server.enableXsrfProtection=false"]
)
time.sleep(4)
print("✅ Streamlit started")

# Download cloudflared if not already present
import os
if not os.path.exists("cloudflared"):
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "cloudflared"
    ], check=True)
    subprocess.run(["chmod", "+x", "cloudflared"])
print("✅ cloudflared ready")

# Start tunnel
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT  # ✅ merge both streams into one
)

print("⏳ Waiting for tunnel URL...")

# Read all lines until we find the full URL
for line in tunnel.stdout:
    text = line.decode("utf-8").strip()
    # Print raw line so you can see everything
    print(text)
    # Extract URL using regex
    match = re.search(r"https://[\w\-]+\.trycloudflare\.com", text)
    if match:
        url = match.group(0)
        print(f"\n{'='*50}")
        print(f"🚀 Your shareable link: {url}")
        print(f"{'='*50}")
        break

✅ Streamlit started
✅ cloudflared ready
⏳ Waiting for tunnel URL...
2026-04-14T06:02:16Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-14T06:02:16Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-14T06:02:19Z INF +--------------------------------------------------------------------------------------------+
2026-04-14T06:02:19Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-14T06